In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\lenovo\AppData\Local\Temp\ipykernel_5572\2300941456.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader = PyPDFLoader("Medical_Book.pdf")
extracted_data = loader.load()

In [3]:
print(f"Loaded {len(extracted_data)} pages.")

Loaded 4505 pages.


In [4]:
print(extracted_data[0])

page_content='' metadata={'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2006-10-16T20:19:33+02:00', 'moddate': '2016-02-07T11:23:03+07:00', 'source': 'Medical_Book.pdf', 'total_pages': 4505, 'page': 0, 'page_label': 'i'}


In [5]:
from langchain_core.documents import Document

def filter_metadata(docs):
    filtered_docs = []

    for doc in docs:
        filtered_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={
                    "source": doc.metadata["source"],
                    "page": doc.metadata["page"],
                },
            )
        )

    return filtered_docs

In [6]:
minimal_docs = filter_metadata(extracted_data)

In [7]:
# Split the documents into smaller chunks

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [8]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 23495


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cuda"},      
        encode_kwargs={
            "normalize_embeddings": True
        },
    )

    return embeddings


embedding = download_embeddings()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\lenovo\.conda\envs\genai\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
print(type(embedding))

<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>


In [11]:
vector = embedding.embed_query("What is diabetes?")
print(len(vector))

384


In [15]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [36]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [37]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

In [18]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [19]:
pc

In [20]:
indexes = pc.list_indexes()

print(indexes)

[]


In [21]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [22]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [23]:
# Load Existing index 

# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)


In [24]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [26]:
retrieved_docs = retriever.invoke("What is Acne?")

In [27]:
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Source: {doc.metadata['source']}")
    print(doc.page_content[:500])  # First 500 characters
    print("-" * 80)

Document 1
Page: 269.0
Source: Medical_Book.pdf
acne. Isotretinoin (Accutane) is prescribed only for
very severe, disfiguring acne.
Acne is a skin condition that occurs when pores or
hair follicles become blocked. This blockage allows a
waxy material called sebum to collect inside the pores
or follicles. Normally, sebum flows out onto the skin
240 GALE ENCYCLOPEDIA OF MEDICINE
Antiacne drugs
--------------------------------------------------------------------------------
Document 2
Page: 55.0
Source: Medical_Book.pdf
skin that houses the oil-producing glands and hair
follicles, where pimples form.
Sebum— An oily skin moisturizer produced by
sebaceous glands.
Tretinoin— A drug that works by increasing the
turnover (death and replacement) of skin cells.
Acne vulgaris affecting a woman’s face. Acne is the general
name given to a skin disorder in which the sebaceous glands
become inflamed. (Photograph by Biophoto Associates, Photo
Researchers, Inc. Reproduced by permission.)
26 GALE ENCYCL

In [38]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=HF_TOKEN,
    max_new_tokens=512,
    temperature=0.2,
)

chatModel = ChatHuggingFace(llm=llm)

In [39]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

In [40]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are a medical assistant for question-answering tasks.

Use only the provided context to answer the user's question.
If the answer cannot be found in the context, say "I don't know based on the provided context."
Keep the answer concise (maximum three sentences).

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [41]:
question_answer_chain = create_stuff_documents_chain(
    chatModel,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

In [42]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of growth hormone from the pituitary gland, leading to excessive growth in bone and soft tissue. In children, this can result in gigantism, characterized by unusual height due to continued bone growth, while in adults, it is called acromegaly, affecting bone and soft tissue growth after growth plates have closed.


In [43]:
response = rag_chain.invoke({"input": "What is Acne?"})

print(response["answer"])

print("\nRetrieved Sources:")
for doc in response["context"]:
    print(f"Page: {doc.metadata.get('page')}")
    print(doc.page_content[:300])
    print("-" * 80)

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.

Retrieved Sources:
Page: 269.0
acne. Isotretinoin (Accutane) is prescribed only for
very severe, disfiguring acne.
Acne is a skin condition that occurs when pores or
hair follicles become blocked. This blockage allows a
waxy material called sebum to collect inside the pores
or follicles. Normally, sebum flows out onto the skin
24
--------------------------------------------------------------------------------
Page: 55.0
skin that houses the oil-producing glands and hair
follicles, where pimples form.
Sebum— An oily skin moisturizer produced by
sebaceous glands.
Tretinoin— A drug that works by increasing the
turnover (death and replacement) of skin cells.
Acne vulgaris affecting a woman’s face. Acne is the general
n
--------------------------------------------------------------------------------
Page: 54.0
exam

In [44]:
response = rag_chain.invoke({"input": "what is the Treatment of Acne?"})
print(response["answer"])

Acne treatment involves reducing sebum production, removing dead skin cells, and killing bacteria using topical drugs and oral medications. The choice of treatment depends on the severity of the acne, ranging from mild to severe.
